In [2]:
import sys, os, re
sys.path.insert(0, os.path.abspath('../src'))

from utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType

spark = get_spark()

BRONZE_CMS   = '../data/bronze/cms_cost_reports'
BRONZE_USDA  = '../data/bronze/usda_rucc'
BRONZE_SHEPS = '../data/bronze/sheps_closures'
SILVER_PATH  = '../data/silver'

df_cms      = spark.read.format('delta').load(BRONZE_CMS)
df_rucc     = spark.read.format('delta').load(BRONZE_USDA)
df_closures = spark.read.format('delta').load(BRONZE_SHEPS)

print(f'CMS rows:          {df_cms.count()}')
print(f'RUCC rows:         {df_rucc.count()}')
print(f'Closure records:   {df_closures.count()}')

# Sanity check — print the actual column names so we know what we are working with.
# The CMS PUF uses title-case names with spaces (e.g. 'State Code', 'Provider CCN').
# We will sanitize them in the next cell before touching any data.
print('\nCMS columns (first 20):')
print(df_cms.columns[:20])

print('\nRUCC columns:')
print(df_rucc.columns)

print('\nSheps columns:')
print(df_closures.columns)

CMS rows:          80077
RUCC rows:         3234
Closure records:   110

CMS columns (first 20):
['rpt_rec_num', 'Provider_CCN', 'Hospital_Name', 'Street_Address', 'City', 'State_Code', 'Zip_Code', 'County', 'Medicare_CBSA_Number', 'Rural_Versus_Urban', 'CCN_Facility_Type', 'Provider_Type', 'Type_of_Control', 'Fiscal_Year_Begin_Date', 'Fiscal_Year_End_Date', 'FTE_-_Employees_on_Payroll', 'Number_of_Interns_and_Residents_FTE', 'Total_Days_Title_V', 'Total_Days_Title_XVIII', 'Total_Days_Title_XIX']

RUCC columns:
['FIPS', 'State', 'County_Name', 'Population_2010', 'RUCC_2013', 'Description']

Sheps columns:
['hospital_name', 'city', 'state', 'closure_year']


In [3]:
# Run this once to identify the financial column names before we sanitize.
# We need to know the exact names for the Imputer in Cell 5.
print(f'Total CMS columns: {len(df_cms.columns)}')
print('\nAll CMS columns:')
for i, col in enumerate(df_cms.columns):
    print(f'  {i:3d}: {col}')

Total CMS columns: 118

All CMS columns:
    0: rpt_rec_num
    1: Provider_CCN
    2: Hospital_Name
    3: Street_Address
    4: City
    5: State_Code
    6: Zip_Code
    7: County
    8: Medicare_CBSA_Number
    9: Rural_Versus_Urban
   10: CCN_Facility_Type
   11: Provider_Type
   12: Type_of_Control
   13: Fiscal_Year_Begin_Date
   14: Fiscal_Year_End_Date
   15: FTE_-_Employees_on_Payroll
   16: Number_of_Interns_and_Residents_FTE
   17: Total_Days_Title_V
   18: Total_Days_Title_XVIII
   19: Total_Days_Title_XIX
   20: Total_Days_V_+_XVIII_+_XIX_+_Unknown
   21: Number_of_Beds
   22: Total_Bed_Days_Available
   23: Total_Discharges_Title_V
   24: Total_Discharges_Title_XVIII
   25: Total_Discharges_Title_XIX
   26: Total_Discharges_V_+_XVIII_+_XIX_+_Unknown
   27: Number_of_Beds_+_Total_for_all_Subproviders
   28: Hospital_Total_Days_Title_V_For_Adults_&_Peds
   29: Hospital_Total_Days_Title_XVIII_For_Adults_&_Peds
   30: Hospital_Total_Days_Title_XIX_For_Adults_&_Peds
   31: Ho

In [4]:
# The sanitize function doesn't change — it handles mixed case + underscores
# + special characters (like 'FTE_-_Employees_on_Payroll') uniformly.
# After this runs, 'Provider_CCN' → 'provider_ccn', 'State_Code' → 'state_code',
# 'County' → 'county', 'RUCC_2013' → 'rucc_2013', and so on.

def sanitize_columns(df):
    """
    Rename all columns to lowercase snake_case.
    Replaces any non-alphanumeric characters with underscores.
    Strips leading/trailing underscores from the result.
    """
    sanitized = []
    for col in df.columns:
        new_col = re.sub(r'[^a-z0-9]+', '_', col.strip().lower()).strip('_')
        sanitized.append(new_col)

    for old, new in zip(df.columns, sanitized):
        if old != new:
            df = df.withColumnRenamed(old, new)
    return df

df_cms      = sanitize_columns(df_cms)
df_rucc     = sanitize_columns(df_rucc)
df_closures = sanitize_columns(df_closures)

print('CMS columns after sanitization:')
for col in df_cms.columns:
    print(f'  {col}')

CMS columns after sanitization:
  rpt_rec_num
  provider_ccn
  hospital_name
  street_address
  city
  state_code
  zip_code
  county
  medicare_cbsa_number
  rural_versus_urban
  ccn_facility_type
  provider_type
  type_of_control
  fiscal_year_begin_date
  fiscal_year_end_date
  fte_employees_on_payroll
  number_of_interns_and_residents_fte
  total_days_title_v
  total_days_title_xviii
  total_days_title_xix
  total_days_v_xviii_xix_unknown
  number_of_beds
  total_bed_days_available
  total_discharges_title_v
  total_discharges_title_xviii
  total_discharges_title_xix
  total_discharges_v_xviii_xix_unknown
  number_of_beds_total_for_all_subproviders
  hospital_total_days_title_v_for_adults_peds
  hospital_total_days_title_xviii_for_adults_peds
  hospital_total_days_title_xix_for_adults_peds
  hospital_total_days_v_xviii_xix_unknown_for_adults_peds
  hospital_number_of_beds_for_adults_peds
  hospital_total_bed_days_available_for_adults_peds
  hospital_total_discharges_title_v_for_adu

In [5]:
# ── WHY NOT FIPS CONSTRUCTION ────────────────────────────────────────────────
# The reference doc assumed CMS had a numeric county code column. The actual
# CMS PUF stores county as a text name (e.g. 'LOS ANGELES'). We cannot
# construct a 5-digit FIPS by concatenating it with state code.
#
# Solution: normalize both county name columns to a common format
# (uppercase, no 'County'/'Parish'/'Borough' suffix, stripped whitespace),
# then join on (state, normalized_county_name).
#
# This join covers the vast majority of hospitals. A small number of counties
# with unusual naming will result in null RUCC — handled by the filter below.
# ────────────────────────────────────────────────────────────────────────────

# ── STEP 1: Normalize CMS county column ─────────────────────────────────────
# CMS 'county' is already uppercase text (e.g. 'LOS ANGELES', 'TRAVIS')
# Some entries may include the word 'COUNTY' — strip it to standardize.
df_cms = df_cms.withColumn(
    'county_normalized',
    F.trim(
        F.regexp_replace(
            F.upper(F.trim(F.col('county'))),
            r'\s*(COUNTY|PARISH|BOROUGH|MUNICIPALITY|CENSUS AREA)\s*$',
            ''
        )
    )
)

# ── STEP 2: Normalize RUCC county_name column ────────────────────────────────
# RUCC 'county_name' uses format 'Los Angeles County', 'Travis County', etc.
df_rucc = df_rucc.withColumn(
    'county_normalized',
    F.trim(
        F.regexp_replace(
            F.upper(F.trim(F.col('county_name'))),
            r'\s*(COUNTY|PARISH|BOROUGH|MUNICIPALITY|CENSUS AREA)\s*$',
            ''
        )
    )
)

# Rename RUCC 'state' to avoid ambiguity on the join
df_rucc = df_rucc.withColumnRenamed('state', 'rucc_state')

# ── STEP 3: Join RUCC onto CMS via state + normalized county name ────────────
# Left join: every CMS row is kept.
# Hospitals whose county name doesn't match RUCC get null rucc_2013.
df_joined = df_cms.join(
    df_rucc.select('rucc_state', 'county_normalized', 'rucc_2013', 'fips'),
    on=[
        df_cms['state_code'] == df_rucc['rucc_state'],
        df_cms['county_normalized'] == df_rucc['county_normalized']
    ],
    how='left'
)

# ── STEP 4: Filter to rural hospitals (RUCC >= 4) ────────────────────────────
# Hospitals where RUCC didn't match (null) are excluded by this filter —
# they cannot be confirmed as rural so we conservatively drop them.
df_rural = df_joined.filter(F.col('rucc_2013') >= 4)

total     = df_cms.count()
rural     = df_rural.count()
no_match  = df_joined.filter(F.col('rucc_2013').isNull()).count()

print(f'All CMS hospital-years:           {total}')
print(f'No RUCC match (null, excluded):   {no_match}  ({no_match/total:.1%})')
print(f'Rural hospitals (RUCC >= 4):      {rural}   ({rural/total:.1%})')

# EXPECTED: ~25-35% rural. A small null% (5-10%) is normal — county name
# formatting differences account for most mismatches.
# If null% is very high (>30%), check that state_code values in CMS
# (2-letter abbreviation like 'CA') match the 'state' column in RUCC.

All CMS hospital-years:           80077
No RUCC match (null, excluded):   7514  (9.4%)
Rural hospitals (RUCC >= 4):      24976   (31.2%)


In [14]:
# ── STEP 1: Parse fiscal year ────────────────────────────────────────────────
df_rural = df_rural.withColumn(
    'fiscal_year',
    F.year(F.to_date(F.col('fiscal_year_end_date'), 'MM/dd/yyyy'))
)

# ── STEP 2: Name normalization helper ────────────────────────────────────────
# Expands common Sheps abbreviations to match CMS full-word spellings,
# then strips all remaining punctuation.
# Each .withColumn() call applies one substitution in sequence.
def normalize_name(df, input_col, output_col):
    """
    Creates a normalized name column on df.
    Expands abbreviations and strips punctuation.
    Returns the modified DataFrame.
    """
    return (
        df
        .withColumn(output_col, F.upper(F.trim(F.col(input_col))))
        .withColumn(output_col, F.regexp_replace(F.col(output_col), r'\bST\.?\b',    'SAINT'))
        .withColumn(output_col, F.regexp_replace(F.col(output_col), r'\bMED CTR\b',  'MEDICAL CENTER'))
        .withColumn(output_col, F.regexp_replace(F.col(output_col), r'\bMED\b',      'MEDICAL'))
        .withColumn(output_col, F.regexp_replace(F.col(output_col), r'\bCTR\b',      'CENTER'))
        .withColumn(output_col, F.regexp_replace(F.col(output_col), r'\bHOSP\b',     'HOSPITAL'))
        .withColumn(output_col, F.regexp_replace(F.col(output_col), r'\bREGL\b',     'REGIONAL'))
        .withColumn(output_col, F.regexp_replace(F.col(output_col), r'\bREG\b',      'REGIONAL'))
        .withColumn(output_col, F.regexp_replace(F.col(output_col), r'\bMEM\b',      'MEMORIAL'))
        .withColumn(output_col, F.regexp_replace(F.col(output_col), r'\bCOMM\b',     'COMMUNITY'))
        .withColumn(output_col, F.regexp_replace(F.col(output_col), r'\bGEN\b',      'GENERAL'))
        .withColumn(output_col, F.regexp_replace(F.col(output_col), r'[^A-Z0-9\s]', ''))
    )

# ── STEP 3: Prepare Sheps with full-name key and 4-word prefix key ───────────
df_closures_clean = df_closures.withColumn(
    'closure_year', F.col('closure_year').cast(IntegerType())
)
df_closures_clean = normalize_name(df_closures_clean, 'hospital_name', 'name_key')
df_closures_clean = df_closures_clean \
    .withColumn('state_key', F.upper(F.trim(F.col('state')))) \
    .withColumn(
        'name_prefix_key',
        F.concat_ws(' ',
            F.slice(
                F.split(F.regexp_replace(
                    F.upper(F.trim(F.col('hospital_name'))), r'[^A-Z0-9\s]', ''
                ), ' '),
            1, 4)
        )
    ) \
    .select('name_key', 'name_prefix_key', 'state_key', 'closure_year')

# ── STEP 4: Same normalization on CMS hospital_name ──────────────────────────
df_rural = normalize_name(df_rural, 'hospital_name', 'name_key')
df_rural = df_rural \
    .withColumn('state_key', F.upper(F.trim(F.col('state_code')))) \
    .withColumn(
        'name_prefix_key',
        F.concat_ws(' ',
            F.slice(
                F.split(F.regexp_replace(
                    F.upper(F.trim(F.col('hospital_name'))), r'[^A-Z0-9\s]', ''
                ), ' '),
            1, 4)
        )
    )

# ── STEP 5: Pass 1 — exact normalized full-name join ─────────────────────────
df_pass1 = df_rural.join(
    df_closures_clean.select('name_key', 'state_key', 'closure_year')
                     .withColumnRenamed('closure_year', 'closure_year_p1'),
    on=['name_key', 'state_key'],
    how='left'
)

# ── STEP 6: Pass 2 — 4-word prefix join ──────────────────────────────────────
df_pass2 = df_rural.join(
    df_closures_clean.select('name_prefix_key', 'state_key', 'closure_year')
                     .withColumnRenamed('closure_year', 'closure_year_p2')
                     .distinct(),
    on=['name_prefix_key', 'state_key'],
    how='left'
)

# ── STEP 7: Merge passes — pass 1 takes priority ─────────────────────────────
df_merged = df_pass1.join(
    df_pass2.select('provider_ccn', 'fiscal_year', 'closure_year_p2'),
    on=['provider_ccn', 'fiscal_year'],
    how='left'
)

df_merged = df_merged.withColumn(
    'closure_year',
    F.coalesce(F.col('closure_year_p1'), F.col('closure_year_p2'))
)

# ── STEP 8: Binary distress label ────────────────────────────────────────────
df_labeled = df_merged.withColumn(
    'distress_label',
    F.when(
        (F.col('closure_year').isNotNull()) &
        (F.col('closure_year') > F.col('fiscal_year')) &
        (F.col('closure_year') - F.col('fiscal_year') <= 3),
        1
    ).otherwise(0).cast(IntegerType())
)

print('Label distribution after two-pass matching:')
df_labeled.groupBy('distress_label').count().show()

total = df_labeled.count()
pos   = df_labeled.filter('distress_label = 1').count()
print(f'Distress rate:            {pos / total:.3f}')
print(f'Unique matched hospitals: '
      f'{df_labeled.filter("distress_label = 1").select("provider_ccn").distinct().count()}')


Label distribution after two-pass matching:
+--------------+-----+
|distress_label|count|
+--------------+-----+
|             1|   87|
|             0|25581|
+--------------+-----+



Distress rate:            0.003
Unique matched hospitals: 32


In [16]:
# ── IMPORTANT: Update this list based on your Cell 1b output ─────────────────
# The names below are the most common CMS PUF financial column names after
# sanitization. Verify each one exists in your df_cms.columns output.
# If a column is named differently, swap in the correct name here.
FINANCIAL_COLS = [
    'total_patient_revenue',
    'net_patient_revenue',
    'total_costs',
    'net_income',
    'less_total_operating_expense',
    'number_of_beds',
    'inpatient_total_charges',
    'medicaid_charges',
    'total_days_title_xviii',
    'total_days_title_xix',
    'cash_on_hand_and_in_banks',
    'total_current_assets',
    'total_current_liabilities',
    'depreciation_cost',
    'cost_of_charity_care',
    'total_bad_debt_expense'
]

# Verify all expected columns exist before proceeding
missing_cols = [c for c in FINANCIAL_COLS if c not in df_labeled.columns]
if missing_cols:
    print(f'WARNING — these columns were not found: {missing_cols}')
    print('Check your Cell 1b output and update FINANCIAL_COLS above.')
else:
    print('All expected financial columns found. Proceeding.')

# ── STEP 1: Cast to DoubleType ────────────────────────────────────────────────
for col_name in FINANCIAL_COLS:
    df_labeled = df_labeled.withColumn(
        col_name,
        F.col(col_name).cast(DoubleType())
    )

# ── STEP 2: Remove rows with nonsensical values ───────────────────────────────
df_labeled = df_labeled.filter(
    (F.col('total_patient_revenue') > 0) | F.col('total_patient_revenue').isNull()
)
df_labeled = df_labeled.filter(
    (F.col('number_of_beds') > 0) | F.col('number_of_beds').isNull()
)

print(f'Rows after removing nonsensical values: {df_labeled.count()}')

# ── STEP 3: Median imputation ─────────────────────────────────────────────────
from pyspark.ml.feature import Imputer

imputer = Imputer(
    inputCols=FINANCIAL_COLS,
    outputCols=FINANCIAL_COLS,
    strategy='median'
)

imputer_model = imputer.fit(df_labeled)
df_silver = imputer_model.transform(df_labeled)

# ── STEP 4: Verify no nulls remain ───────────────────────────────────────────
null_counts = df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in FINANCIAL_COLS
]).collect()[0]

print('\nRemaining nulls per financial column (all should be 0):')
for col_name in FINANCIAL_COLS:
    print(f'  {col_name}: {null_counts[col_name]}')

All expected financial columns found. Proceeding.
Rows after removing nonsensical values: 25664

Remaining nulls per financial column (all should be 0):
  total_patient_revenue: 0
  net_patient_revenue: 0
  total_costs: 0
  net_income: 0
  less_total_operating_expense: 0
  number_of_beds: 0
  inpatient_total_charges: 0
  medicaid_charges: 0
  total_days_title_xviii: 0
  total_days_title_xix: 0
  cash_on_hand_and_in_banks: 0
  total_current_assets: 0
  total_current_liabilities: 0
  depreciation_cost: 0
  cost_of_charity_care: 0
  total_bad_debt_expense: 0


In [19]:
KEEP_COLS = [
    'provider_ccn',
    'fiscal_year',
    'fips',
    'state_code',
    'rucc_2013',
    'number_of_beds',
    'total_patient_revenue',
    'net_patient_revenue',
    'total_costs',
    'net_income',
    'less_total_operating_expense',
    'medicaid_charges',
    'inpatient_total_charges',
    'total_days_title_xviii',
    'total_days_title_xix',
    'cash_on_hand_and_in_banks',
    'total_current_assets',
    'total_current_liabilities',
    'depreciation_cost',
    'cost_of_charity_care',
    'total_bad_debt_expense',
    'distress_label'
]

df_silver_final = (
    df_silver
    .select(KEEP_COLS)
    .withColumnRenamed('provider_ccn', 'ccn_key')
    # Cast ccn_key to zero-padded 6-character string
    # An integer CCN of 10001 must become '010001' — without this,
    # the Gold layer join on ccn_key will silently miss every hospital
    # whose CCN has a leading zero
    .withColumn('ccn_key', F.lpad(F.col('ccn_key').cast(StringType()), 6, '0'))
    # Cast fips to zero-padded 5-character string for the same reason
    .withColumn('fips', F.lpad(F.col('fips').cast(StringType()), 5, '0'))
)

(
    df_silver_final
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(SILVER_PATH)
)

df_check   = spark.read.format('delta').load(SILVER_PATH)
total_rows = df_check.count()
distress_1 = df_check.filter('distress_label = 1').count()

print(f'Silver layer written to:  {SILVER_PATH}')
print(f'Total rows:               {total_rows}')
print(f'Distress rows (label=1):  {distress_1}')
print(f'Distress rate:            {distress_1 / total_rows:.3f}')
df_check.printSchema()

Silver layer written to:  ../data/silver
Total rows:               25664
Distress rows (label=1):  87
Distress rate:            0.003
root
 |-- ccn_key: string (nullable = true)
 |-- fiscal_year: integer (nullable = true)
 |-- fips: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- rucc_2013: double (nullable = true)
 |-- number_of_beds: double (nullable = true)
 |-- total_patient_revenue: double (nullable = true)
 |-- net_patient_revenue: double (nullable = true)
 |-- total_costs: double (nullable = true)
 |-- net_income: double (nullable = true)
 |-- less_total_operating_expense: double (nullable = true)
 |-- medicaid_charges: double (nullable = true)
 |-- inpatient_total_charges: double (nullable = true)
 |-- total_days_title_xviii: double (nullable = true)
 |-- total_days_title_xix: double (nullable = true)
 |-- cash_on_hand_and_in_banks: double (nullable = true)
 |-- total_current_assets: double (nullable = true)
 |-- total_current_liabilities: double (nullab